# CLOAK IndieGoGo Campaign Simulator
**Product:** CLOAK Electronic Safe Keypad Shield â€" $179.99
**Platform:** IndieGoGo (all-or-nothing, post-Gamefound)
**Method:** Monte Carlo simulation over marketing funnel (10,000 runs)
**Purpose:** Risk assessment, gap analysis, and pre-launch planning

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import replace
from IPython.display import display, Markdown

from src.config import CloakConfig, CampaignConfig, FeeStructure
from src.simulation import SimulationInputs, run_simulation
from src.gap import gap_analysis
from src.sensitivity import tornado_analysis
from src.demand import demand_confidence_score, VALIDATION_PLAYBOOK
from src.market import (
    US_GUN_OWNERS, SAFE_OWNERSHIP_RATE, ELECTRONIC_KEYPAD_RATE,
    ADDRESSABLE_SAFE_OWNERS, META_TARGETABLE_AUDIENCE, saturation_check,
)
from src.data_loader import KS_CSV, IGG_COMPARABLES
from src.viz import (
    campaign_scorecard, funding_trajectory_fan_chart, gap_analysis_table,
    tornado_chart, demand_signal_dashboard, scenario_comparison_table,
)

cloak = CloakConfig(standard_price=179.99, early_bird_price=149.99, early_bird_quantity=50, cogs_per_unit=45.00, shipping_per_unit=12.00)
campaign = CampaignConfig(goal=15_000, duration_days=30)
fees = FeeStructure()
SEED = 42
N_RUNS = 10_000

print(f"CLOAK: ${cloak.standard_price} (early bird: ${cloak.early_bird_price} x {cloak.early_bird_quantity})")
print(f"Campaign: ${campaign.goal:,.0f} goal, {campaign.duration_days} days")
print(f"Fees: {fees.platform_rate:.0%} platform + {fees.processing_rate:.0%} processing + ${fees.per_txn_fee}/txn")
print(f"Simulation: {N_RUNS:,} Monte Carlo runs, seed={SEED}")

## Data Foundations
Kickstarter data (378K campaigns) provides the statistical base. Filtered to comparable campaigns (Technology/Design, $5K-$100K goal, $50-$500 avg pledge), then used as reference benchmarks. IndieGoGo comparables serve as validation.

In [ ]:
from src.data_loader import load_kickstarter, filter_kickstarter_comparables, kickstarter_stats, load_igg_comparables, igg_comparables_stats

if KS_CSV.exists():
    ks_raw = load_kickstarter()
    ks_filtered = filter_kickstarter_comparables(ks_raw)
    ks_stats = kickstarter_stats(ks_filtered)

    print(f"=== KICKSTARTER DATA (Tier 1) ===\n")
    print(f"Total campaigns loaded:     {len(ks_raw):>10,}")
    print(f"Comparable campaigns:       {ks_stats['total_campaigns']:>10,}")
    print(f"Success rate (comparable):  {ks_stats['success_rate']:>10.1%}")
    print(f"Median backers (success):   {ks_stats['median_backers_successful']:>10,}")
    print(f"Median raised (success):    ${ks_stats['median_raised_successful']:>10,.0f}")
    print(f"Median goal:                ${ks_stats['median_goal']:>10,.0f}")
    print(f"Median avg pledge:          ${ks_stats['avg_pledge_median']:>10,.2f}")
    print(f"Backers 10th-90th pctl:     {ks_stats['backers_p10']:,} - {ks_stats['backers_p90']:,}")
    print(f"\nKey takeaway: comparable campaigns succeed 43% of the time with a")
    print(f"median of 245 backers and $30K raised. CLOAK needs ~84 backers at")
    print(f"$179.99 to hit its $15K goal -- well within the successful range.")
else:
    print("Kickstarter CSV not found. Run: kaggle datasets download -d kemical/kickstarter-projects -p data/kickstarter/ --unzip")

if IGG_COMPARABLES.exists():
    igg_comps = load_igg_comparables()
    igg_stats = igg_comparables_stats(igg_comps)
    print(f"\n=== INDIEGOGO COMPARABLES (Tier 2) ===")
    print(f"Campaigns researched: {igg_stats['count']}")
    if igg_stats["count"] > 1:
        print(f"Success rate: {igg_stats['success_rate']:.1%}")
        print(f"Median raised: ${igg_stats['median_raised']:,.0f}")
    else:
        print("(Placeholder data -- research needed for 10-20 comparable campaigns)")

## Market Sizing â€" Realism Ceiling
The addressable market sets the upper bound on what's achievable.

In [ ]:
print("=== ADDRESSABLE MARKET ===\n")
print(f"US gun owners:              {US_GUN_OWNERS.value:>14,}  [{US_GUN_OWNERS.source}]")
print(f"Safe ownership rate:        {SAFE_OWNERSHIP_RATE.value:>14.0%}  [{SAFE_OWNERSHIP_RATE.source}]")
print(f"Electronic keypad rate:     {ELECTRONIC_KEYPAD_RATE.value:>14.0%}  [{ELECTRONIC_KEYPAD_RATE.source}]")
print(f"{'â"€' * 60}")
print(f"Addressable safe owners:    {ADDRESSABLE_SAFE_OWNERS.value:>14,}  [Tier {ADDRESSABLE_SAFE_OWNERS.tier}]")
print(f"Meta targetable audience:   {META_TARGETABLE_AUDIENCE.value:>14,}  [Tier {META_TARGETABLE_AUDIENCE.tier}]")

## Demand Signal Analysis â€" Go/No-Go Gate
Before the simulation matters, we need to assess evidence of product-market fit.

In [ ]:
demand = demand_confidence_score()
print(f"=== DEMAND CONFIDENCE: {demand['rating'].upper()} ===\n")
print(f"Score: {demand['total_score']}/{demand['max_possible']}")
print(f"  Direct signals:   {demand['direct_score']}/12")
print(f"  Indirect signals: {demand['indirect_score']}/15")
print(f"  Category signals: {demand['category_score']}/9")
print(f"\n{demand['narrative']}\n")
display(demand_signal_dashboard(demand["signals"], demand["rating"], demand["narrative"]))
print("\n=== PRE-LAUNCH VALIDATION PLAYBOOK ===\n")
for i, step in enumerate(VALIDATION_PLAYBOOK, 1):
    print(f"{i}. {step['action']}")
    print(f"   Cost: {step['cost']} | Effort: {step['effort']} | When: {step['timeline']}")
    print(f"   Impact: {step['impact']}\n")

## Audience Inputs â€" What You Have
Change these values to match the CLOAK team's current state, then re-run the notebook.

In [ ]:
inputs = SimulationInputs(
    email_list=50,             # Family, friends, friends-of-friends
    ig_followers=124,          # Instagram followers (from Meta data)
    fb_followers=69,           # Facebook followers (from Meta data)
    daily_ad_budget=0.0,       # Daily paid ads budget ($)
    pr_hits=0,                 # Expected press/media articles
    monthly_site_visitors=100, # Monthly unique visitors to bosscoversusa.com
    cloak=cloak, campaign=campaign, fees=fees,
)

print("Running simulation...")
results = run_simulation(inputs, n_runs=N_RUNS, seed=SEED)
prob = results.probability_of_funding()
raised_pcts = {p: float(results.percentiles([p])[0]) for p in [10, 50, 90]}
backer_pcts = {p: float(np.percentile(results.total_backers, p)) for p in [10, 50, 90]}
median_net = float(np.median(results.net_revenue))
print(campaign_scorecard(prob, raised_pcts, backer_pcts, median_net, demand["rating"], campaign.goal))

In [ ]:
fig = funding_trajectory_fan_chart(results.daily_trajectories, campaign.goal, campaign.duration_days)
plt.show()

## Gap Analysis â€" What You Need
For each audience channel: what you have, what you'd need to reach 70% confidence, and whether that's realistic.

In [ ]:
print("Running gap analysis (this takes ~60 seconds)...")
gap_70 = gap_analysis(inputs, target_probability=0.70, n_runs=300, seed=SEED)
for channel, data in gap_70.items():
    check = saturation_check(data["you_need"] if isinstance(data["you_need"], int) else 0, channel)
    data["market_check"] = check["feasibility"]
display(gap_analysis_table(gap_70, target_pct=70))

## Sensitivity Analysis â€" Where to Focus
Which levers have the biggest impact on campaign outcomes?

In [ ]:
print("Running sensitivity analysis...")
tornado_data = tornado_analysis(inputs, n_runs=500, seed=SEED)
baseline = float(np.median(results.total_raised))
fig = tornado_chart(tornado_data, baseline)
plt.show()
print("\nImpact ranking:")
for channel, data in tornado_data.items():
    print(f"  {channel.replace('_', ' ').title():.<30s} ${data['impact']:>10,.0f} impact range")

## Scenario Comparison
Four scenarios showing how different levels of pre-launch preparation affect outcomes.

In [ ]:
scenarios_config = [
    {"name": "Current State", "overrides": {}},
    {"name": "Modest Prep (8 weeks)", "overrides": {"email_list": 500, "daily_ad_budget": 20.0, "pr_hits": 1}},
    {"name": "Strong Prep (12 weeks)", "overrides": {"email_list": 2000, "daily_ad_budget": 75.0, "pr_hits": 3, "ig_followers": 500}},
    {"name": "Ideal Case", "overrides": {"email_list": 5000, "daily_ad_budget": 100.0, "pr_hits": 5, "ig_followers": 2000, "fb_followers": 500, "monthly_site_visitors": 1000}},
]

# Run simulation once per scenario and cache results
scenario_results = []
for sc in scenarios_config:
    sc_inputs = replace(inputs, **sc["overrides"])
    sc_results = run_simulation(sc_inputs, n_runs=N_RUNS, seed=SEED)
    scenario_results.append({
        "name": sc["name"],
        "inputs": sc_inputs,
        "results": sc_results,
        "prob": sc_results.probability_of_funding(),
        "median_raised": float(np.median(sc_results.total_raised)),
        "median_net": float(np.median(sc_results.net_revenue)),
    })

display(scenario_comparison_table(scenario_results))

fig, ax = plt.subplots(figsize=(12, 6))
colors = ["#e74c3c", "#f39c12", "#27ae60", "#2980b9"]
for i, sc in enumerate(scenario_results):
    p50 = np.percentile(sc["results"].daily_trajectories, 50, axis=0)
    days = np.arange(1, campaign.duration_days + 1)
    ax.plot(days, p50, color=colors[i], linewidth=2, label=sc["name"])
ax.axhline(y=campaign.goal, color="red", linestyle="--", alpha=0.5, label=f"Goal: ${campaign.goal:,.0f}")
ax.set_xlabel("Campaign Day")
ax.set_ylabel("Cumulative Funds Raised ($)")
ax.set_title("Scenario Comparison -- Median Funding Trajectories")
ax.legend()
plt.tight_layout()
plt.show()


## Recommendations & Next Steps

In [ ]:
print("=" * 60)
print("  RECOMMENDATIONS")
print("=" * 60)
if demand["rating"] == "Weak":
    print("\n  DEMAND CONFIDENCE IS WEAK")
    print("  Before committing to a campaign, complete the validation")
    print("  playbook above. The simulation shows what's POSSIBLE,")
    print("  not what's PROBABLE without proven demand.\n")
print("RANKED PRE-LAUNCH ACTIONS BY SIMULATION IMPACT:\n")
actions = [
    ("Build email list to 500+", "Moves P(funded) from current to modest scenario", "High"),
    ("Run $20/day Facebook ads for 2 weeks", "Provides real conversion data AND builds audience", "High"),
    ("Create IndieGoGo pre-launch page", "Free demand signal + builds early backer list", "Medium"),
    ("Increase Instagram posting to 3x/week", "Leverage existing 124 followers + organic reach", "Medium"),
    ("Secure 1-3 press/media mentions", "High-variance but potentially high-impact", "Medium"),
    ("Post in gun safe forums/communities", "Free demand validation + audience building", "Low cost"),
]
for i, (action, impact, priority) in enumerate(actions, 1):
    print(f"  {i}. [{priority}] {action}")
    print(f"     {impact}\n")
print("â"€" * 60)
print(f"\nGO/NO-GO FRAMEWORK:")
print(f"  P(funded) < 20%:  DO NOT LAUNCH without more preparation")
print(f"  P(funded) 20-50%: RISKY â€" consider a lower goal or more prep time")
print(f"  P(funded) 50-70%: VIABLE â€" proceed with active marketing plan")
print(f"  P(funded) > 70%:  STRONG â€" launch with confidence")
print(f"\n  Current P(funded): {prob:.1%}")